In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains

def first_page():
    elements = driver.find_elements(By.CSS_SELECTOR, "a[class='D113_link']")
    return [(e.text, e.get_attribute('href')) for e in elements]

def second_page(link):
    driver.get(link)
    driver.maximize_window()
    time.sleep(2)  # Ensure page is fully loaded
    routes = [(r.text, r.get_attribute('href')) for r in driver.find_elements(By.CSS_SELECTOR, "a[class='route']")]

    # Go through all available pages if present
    pages = driver.find_elements(By.CSS_SELECTOR, "div[class='DC_117_pageTabs ']")
    for p in pages:
        ActionChains(driver).click(p).perform()
        time.sleep(3)
        new_routes = driver.find_elements(By.CSS_SELECTOR, "a[class='route']")
        routes.extend((r.text, r.get_attribute('href')) for r in new_routes)
    return routes

def final_page_scraping(name, link):
    driver.get(link)
    driver.maximize_window()
    time.sleep(1)
    
    # Click on buttons if necessary
    buttons = driver.find_elements(By.CSS_SELECTOR, "div[class='button']")
    for button in reversed(buttons):
        button.click()

    # Scroll to ensure all elements are loaded
    for _ in range(10):  # Adjust the range for longer scrolling
        driver.execute_script('window.scrollBy(0, 1000)')
        time.sleep(0.5)
    
    # Extract bus information
    
    bus_name = driver.find_elements(By.CSS_SELECTOR, "div[class^='travelsName']")
    bus_type = driver.find_elements(By.CSS_SELECTOR, "p[class^='busType']")
    bus_dept = driver.find_elements(By.CSS_SELECTOR, "p[class^='boardingTime']")
    bus_dur = driver.find_elements(By.CSS_SELECTOR, "p[class^='duration']")
    bus_reach = driver.find_elements(By.CSS_SELECTOR, "p[class^='droppingTime']")
    bus_star = driver.find_elements(By.CSS_SELECTOR, "div[class^='rating']")
    bus_price = driver.find_elements(By.CSS_SELECTOR, "p[class^='finalFare']")
    bus_seats = driver.find_elements(By.CSS_SELECTOR, "p[class^='totalSeats']")
    
    if not all([bus_name, bus_type, bus_dept, bus_dur, bus_reach, bus_star, bus_price, bus_seats]):
        print(f"[SKIPPED] Missing data for route: {name}")
        return []


    for i in range(len(bus_name)):
        min_len = min(
            len(bus_name),
            len(bus_type),
            len(bus_dept),
            len(bus_dur),
            len(bus_reach),
            len(bus_star),
            len(bus_price),
            len(bus_seats)
        )
    bus_data = []
    for i in range(min_len):
        bus_data.append([
            name, link,
            bus_name[i].text,
            bus_type[i].text,
            bus_dept[i].text,
            bus_dur[i].text,
            bus_reach[i].text,
            bus_star[i].text,
            bus_price[i].text,
            bus_seats[i].text.split('\n')[0]
        ])

    
    return bus_data

driver = webdriver.Chrome()
driver.get('https://www.redbus.in/online-booking/rtc-directory')
driver.maximize_window()
time.sleep(5)
                                                                                                               
result = []
f_page = first_page()
for name, link in f_page:
    second_page_data = second_page(link)
    for route_name, route_link in second_page_data:
        bus_info = final_page_scraping(route_name, route_link)
        result.extend(bus_info)

# DataFrame creation and saving to Excel
columns = ['Bus Route Name', 'Bus route link', 'Bus Name', 'Bus Type', 'Departing time', 'Duration', 'Reaching Time', 'Rating', 'Price', 'Seats available']
df = pd.DataFrame(result, columns=columns).drop_duplicates()

# Clean and format the DataFrame
df['Departing time'] = pd.to_datetime(df['Departing time'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
df['Reaching Time'] = pd.to_datetime(df['Reaching Time'], errors='coerce').dt.strftime('%Y-%m-%d %H:%M:%S')
df['Price'] = df['Price'].str.replace(r'[^\d.]', '', regex=True).astype(float)
df['Seats available'] = df['Seats available'].str.extract('(\d+)').astype(float)
df['Rating'] = pd.to_numeric(df['Rating'].str.replace('New', '').str.strip(), errors='coerce').fillna(0)

# Save to Excel
df.to_excel("output_redbus.xlsx", index=False)
print("Data extracted and saved to output_redbus.xlsx")

# Close the browser
driver.quit()


[SKIPPED] Missing data for route: Dharamshala (Himachal Pradesh) to Chandigarh


KeyboardInterrupt: 

In [ ]:
df

,Bus Route Name,Bus route link,Bus Name,Bus Type,Departing time,Duration,Reaching Time,Rating,Price,Seats available
0,Hyderabad to Vijayawada,https://www.redbus.in/bus-tickets/hyderabad-to...,FRESHBUS,Electric A/C Seater (2+2),2025-06-20 12:20:00,6h 50m,2025-06-20 19:10:00,4.8,300.0,12.0
1,Hyderabad to Vijayawada,https://www.redbus.in/bus-tickets/hyderabad-to...,FRESHBUS,Electric A/C Seater (2+2),2025-06-20 12:40:00,7h,2025-06-20 19:40:00,613.0,300.0,6.0
2,Hyderabad to Vijayawada,https://www.redbus.in/bus-tickets/hyderabad-to...,IntrCity SmartBus,A/C Seater / Sleeper (2+1),2025-06-20 23:05:00,7h 15m,2025-06-20 06:20:00,4.7,599.0,22.0
3,Hyderabad to Vijayawada,https://www.redbus.in/bus-tickets/hyderabad-to...,IntrCity SmartBus,AC Sleeper (2+1),2025-06-20 00:20:00,6h 25m,2025-06-20 06:45:00,949.0,719.0,22.0
4,Hyderabad to Vijayawada,https://www.redbus.in/bus-tickets/hyderabad-to...,FRESHBUS,Electric A/C Seater (2+2),2025-06-20 12:20:00,5h 35m,2025-06-20 17:55:00,4.6,300.0,16.0
...,...,...,...,...,...,...,...,...,...,...
2780,Siliguri to Singtham (Sikkim),https://www.redbus.in/bus-tickets/siliguri-to-...,Sikkim Nationalised Transport (SNT) - 171241,A/C Seater (2+1),2025-06-20 13:30:00,4h 25m,2025-06-20 17:55:00,3.4,370.0,15.0
2781,Siliguri to Singtham (Sikkim),https://www.redbus.in/bus-tickets/siliguri-to-...,Sikkim Nationalised Transport (SNT) - 170874,NON A/C Seater (2+2),2025-06-20 15:00:00,4h 25m,2025-06-20 19:25:00,17.0,205.0,25.0
2782,Singtham (Sikkim) to Siliguri,https://www.redbus.in/bus-tickets/singtham-sik...,Sikkim Nationalised Transport (SNT) - 171248,A/C Seater (2+1),2025-06-20 13:45:00,4h,2025-06-20 17:45:00,3.9,500.0,5.0
2783,Siliguri to Namchi (Sikkim),https://www.redbus.in/bus-tickets/siliguri-to-...,Sikkim Nationalised Transport (SNT) - 170761,NON A/C Seater (2+2),2025-06-20 13:30:00,4h 30m,2025-06-20 18:00:00,2.6,220.0,22.0
